In [97]:
import spacy
from spacy import displacy
from spacy.tokens import Span
from spacy.training import Example

In [98]:
nlp = spacy.load("en_core_web_sm")
doc = nlp(input())
print(doc.ents)

(Ali, Toyota, Dammam)


In [99]:
for ent in doc.ents:
    print(f"{ent.text}: {ent.label_} ({spacy.explain(ent.label_)}), tokens {ent.start} to {ent.end}")

Ali: PERSON (People, including fictional), tokens 0 to 1
Toyota: ORG (Companies, agencies, institutions, etc.), tokens 2 to 3
Dammam: GPE (Countries, cities, states), tokens 4 to 5


In [100]:
paragraph = "This agreement is made and entered into by and between ‘Abc & Co.’  and ‘Bcd LLC’ for term 1 year starting from April 1, 2020,  hereinafter collectively referred to as the Parties. Samuel P. Jackson in the place (New York) and on the date written below, with the following terms and conditions."
doc = nlp(paragraph)
print(doc.ents)

(between ‘Abc & Co., 1 year, April 1, 2020, Samuel P. Jackson, New York)


In [101]:
doc.ents = [Span(doc, 11, 14, label="ORG"), Span(doc, 18, 20, label="ORG")] + list(doc.ents[1:])
print(doc.ents)

# Bcd LLC was not identified and Abc & Co. was found as "between ‘Abc & Co.", so both are added as ORG

(Abc & Co., Bcd LLC, 1 year, April 1, 2020, Samuel P. Jackson, New York)


In [102]:
labels = [ent.label_ for ent in doc.ents]
print("PERSON:", labels.count("PERSON"))
print("ORG:", labels.count("ORG"))

PERSON: 1
ORG: 2


In [103]:
displacy.render(doc, style="ent", jupyter=True)

In [104]:
text = "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products."
entities = [(0, 9, "ORG"), (25, 35, "PERSON"), (40, 50, "PERSON"), (54, 65, "GPE"), (67, 77, "GPE"), (99, 106, "PRODUCT"), (128, 134, "PRODUCT")]
example = Example.from_dict(nlp.make_doc(text), {"entities": entities})
print(example.reference.ents)

(Microsoft, Bill Gates, Paul Allen, Albuquerque, New Mexico, Windows, Office)


In [105]:
optimizer = nlp.resume_training()

with nlp.select_pipes(enable="ner"):
    for i in range(20):
        losses = {}
        nlp.update([example], drop=0.3, sgd=optimizer, losses=losses)
        print(f"Iteration {i + 1}/20 - Loss: {losses['ner']:.4f}")

Iteration 1/20 - Loss: 3.1317
Iteration 2/20 - Loss: 2.6007
Iteration 3/20 - Loss: 4.6272
Iteration 4/20 - Loss: 2.8569
Iteration 5/20 - Loss: 0.1265
Iteration 6/20 - Loss: 1.7239
Iteration 7/20 - Loss: 2.1173
Iteration 8/20 - Loss: 0.0013
Iteration 9/20 - Loss: 0.0009
Iteration 10/20 - Loss: 0.1938
Iteration 11/20 - Loss: 0.0206
Iteration 12/20 - Loss: 0.0016
Iteration 13/20 - Loss: 0.0177
Iteration 14/20 - Loss: 0.0000
Iteration 15/20 - Loss: 0.0053
Iteration 16/20 - Loss: 0.0005
Iteration 17/20 - Loss: 0.0000
Iteration 18/20 - Loss: 0.0061
Iteration 19/20 - Loss: 0.0000
Iteration 20/20 - Loss: 0.0000


In [106]:
nlp.to_disk("custom_ner_model")
custom_nlp = spacy.load("custom_ner_model")
doc = custom_nlp("Apple was founded by Steve Jobs in Cupertino, California. It is known for its iPhone and Mac products.")

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

Apple -> ORG
Steve Jobs -> PERSON
Cupertino -> GPE
California -> GPE
iPhone -> ORG
Mac -> PRODUCT
